# Qwen3.5-0.8B × CeNN-X2 — middle-layer ablation and distillation

**Purpose:** surgically replace a single **token mixer** of `Qwen/Qwen3.5-0.8B` with **CeNN-X2** and compare a selection of **middle Gated DeltaNet and full-attention layers**. This is a **research experiment**, not a pretrained-model speedup claim.

### Controlled experiment

- Architecture: 24 decoder blocks; linear-attention layers at other indices; full-attention layers **3, 7, 11, 15, 19, 23**.
- Default candidate layers: **8, 10, 11, 12, 14, 15, 16, 19** (both mixer types).
- Architectures: **single-timescale CeNN-X2**, **dual-timescale CeNN-X2**, **dual-timescale + bounded local SDPA**.
- Calibrate **each candidate independently** using native mixer input/output captured from a **frozen** Qwen teacher; preserve RMSNorm, MLP and residual structure.
- Use disjoint **WikiText-2 train / validation / test** splits. Select on validation; audit chosen candidate on test only.
- Report layer cosine/NMSE, held-out next-token NLL, KL/top-1 agreement, *whole-model* prefill latency and peak allocated CUDA memory. Never treat untrained substitution as a quality-preserving comparison.

⚠️ **Limitations:** The graft is for **text-only, fixed-length unpadded sequences**, with `use_cache=False` (full prefill). It is **not** a cache-compatible drop-in for `.generate()`, does not include vision tokens, and is not optimized Triton code. Local SDPA preserves exact *local* retrieval only. A native Gated DeltaNet has highly optimized kernels and may be faster than this prototype.

**Runtime:** Runtime → Change runtime type → **GPU**. A100/L4 preferable for bf16. T4 uses fp16, which requires checking finite outputs. Model weights download from Hugging Face (several GB including downloads); datasets require network. All source for CeNN-X2 is embedded in this notebook, so a checkout is not needed.

In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.34,<2" "datasets>=3,<5" "safetensors" "pandas>=2" "matplotlib>=3.7" "accelerate>=1.0"

In [ ]:
import os, sys, gc, time, json, math, random, hashlib, statistics
from pathlib import Path
from dataclasses import asdict
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoConfig, Qwen3_5ForCausalLM

# @title Experiment settings
MODEL_ID = "Qwen/Qwen3.5-0.8B" # @param {type:"string"}
PROFILE = "quick" # @param ["quick", "research"]
CANDIDATE_LAYERS = [8, 10, 11, 12, 14, 15, 16, 19]
VARIANTS = ["single", "dual", "dual_local"]
SEQ_LEN = 64 # @param {type:"integer"}
BATCH_SIZE = 2 # @param {type:"integer"}
TRAIN_STEPS = 24 if PROFILE == "quick" else 100
TRAIN_BLOCKS = 16 if PROFILE == "quick" else 64
DEV_BLOCKS = 8 if PROFILE == "quick" else 16
TEST_BLOCKS = 8 if PROFILE == "quick" else 24
CHUNK_SIZE = 32
LR = 7e-4
SEED = 1234
OUTPUT_DIR = Path("/content/qwen35_cenn_x2_middle_ablation" if Path('/content').exists() else "./qwen35_cenn_x2_middle_ablation")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = (torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
         else torch.float16 if DEVICE.type == 'cuda' else torch.float32)
print(f"Device={DEVICE}, dtype={DTYPE}, profile={PROFILE}, steps={TRAIN_STEPS}")
if DEVICE.type != 'cuda':
    raise RuntimeError("This full Qwen3.5 comparison requires a CUDA GPU. For CPU only, run the CeNN-X2 contract tests below in a separate session.")
print(torch.cuda.get_device_name(0))

## 1. Embed the CeNN-X2 source
Two files are copied from the prior validated CeNN-X2 notebook; this experiment intentionally does not depend on a change to the pip-installed `tinycenn_lm` package. The underlying reference uses FP32 memory and a bounded chunk loop, **not** fused GPU kernels.

In [ ]:
import sys
MODULE_ROOT=OUTPUT_DIR/'embedded_source'
MODULE_DIR=MODULE_ROOT/'cenn_x2_embedded'
MODULE_DIR.mkdir(parents=True,exist_ok=True)
(MODULE_DIR/'__init__.py').write_text('')
(MODULE_DIR/'cenn_xlstm_gpu.py').write_text('"""Chunk-parallel xLSTM-inspired CeNN mixer (experimental).\n\nIndependent implementation. mLSTM-style stabilized scalar exponential gates,\nkey/value matrix memory, and a depthwise cellular convolution. The recurrence\nhas no tokenwise Python loop: one vectorized GEMM path per fixed-size chunk.\nThis is *not* a drop-in, pretrained xLSTM checkpoint or a fused Triton kernel.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass CeNNXLSTMConfig:\n    dim: int = 256\n    heads: int = 4\n    feature_dim: int = 32\n    value_dim: int = 64\n    chunk_size: int = 64\n    conv_kernel: int = 5\n    bidirectional: bool = False\n\n    def __post_init__(self):\n        if min(self.dim, self.heads, self.feature_dim, self.value_dim) <= 0:\n            raise ValueError("dim, heads and head dimensions must be positive")\n        if not 1 <= self.chunk_size <= 256:\n            raise ValueError("chunk_size must lie in [1, 256]")\n        if self.conv_kernel <= 0 or self.conv_kernel % 2 != 1:\n            raise ValueError("conv_kernel must be a positive odd number")\n\n\n@dataclass\nclass CeNNXLSTMState:\n    # All recurrent state tensors share [batch, heads] leading dimensions.\n    matrix: Tensor                 # [B,H,F,Dv], normalized matrix memory\n    normalizer: Tensor             # [B,H,F], normalized vector memory\n    log_scale: Tensor              # [B,H], log stabilization scale\n    conv_cache: Tensor | None = None  # [B,dim,K-1] in causal mode\n\n    @property\n    def nbytes(self) -> int:\n        return sum(t.numel() * t.element_size() for t in\n                   (self.matrix, self.normalizer, self.log_scale, self.conv_cache)\n                   if t is not None)\n\n\ndef initialize_state(batch: int, heads: int, features: int, values: int,\n                     *, device: torch.device, dtype: torch.dtype) -> CeNNXLSTMState:\n    return CeNNXLSTMState(\n        matrix=torch.zeros(batch, heads, features, values, device=device, dtype=dtype),\n        normalizer=torch.zeros(batch, heads, features, device=device, dtype=dtype),\n        log_scale=torch.zeros(batch, heads, device=device, dtype=dtype),\n    )\n\n\ndef chunkwise_cenn_mlstm(q: Tensor, k: Tensor, v: Tensor, log_forget: Tensor,\n                         log_input: Tensor, state: CeNNXLSTMState | None = None,\n                         *, chunk_size: int = 64) -> tuple[Tensor, CeNNXLSTMState]:\n    """Compute causal stabilized mLSTM memory with GEMMs and per-chunk scans.\n\n    q,k: [B,H,T,F], v: [B,H,T,Dv]; log_forget/log_input: [B,H,T].\n    m_t = max(log(f_t)+m_{t-1}, log(i_t)); C_t and n_t are scaled by exp(-m_t).\n    log_input can be -inf to skip padded positions; log_forget=0 then preserves state.\n    This reference constructs CxC matrices *per chunk*, never TxT globally.\n    State must be in the same dtype/device as inputs (use fp32 for numerical safety).\n    """\n    if not 1 <= chunk_size <= 256:\n        raise ValueError("chunk_size must lie in [1, 256]")\n    if q.ndim != 4 or k.shape != q.shape or v.shape[:3] != q.shape[:3]:\n        raise ValueError("expected q,k [B,H,T,F] and v [B,H,T,Dv]")\n    if q.shape[2] == 0 or log_forget.shape != q.shape[:3] or log_input.shape != q.shape[:3]:\n        raise ValueError("empty sequence or invalid gate shapes")\n    if any(x.dtype != q.dtype or x.device != q.device for x in (k, v, log_forget, log_input)):\n        raise ValueError("all inputs must have equal dtype and device")\n    b, h, t, f = q.shape\n    if state is None:\n        state = initialize_state(b, h, f, v.shape[-1], device=q.device, dtype=q.dtype)\n    c, n, m = state.matrix, state.normalizer, state.log_scale\n    if (c.shape != (b, h, f, v.shape[-1]) or n.shape != (b, h, f)\n            or m.shape != (b, h) or c.dtype != q.dtype or c.device != q.device):\n        raise ValueError("recurrent state mismatch")\n\n    outputs = []\n    for start in range(0, t, chunk_size):\n        stop = min(start + chunk_size, t)\n        qc, kc, vc = q[:, :, start:stop], k[:, :, start:stop], v[:, :, start:stop]\n        lf, li = log_forget[:, :, start:stop], log_input[:, :, start:stop]\n        p = lf.cumsum(dim=-1)                                      # [B,H,C]\n        # m_t = p_t + max(m_previous, max_{j<=t}(log(i_j) - p_j))\n        best = torch.maximum((li - p).cummax(dim=-1).values, m.unsqueeze(-1))\n        mt = p + best\n        old_scale = torch.exp(m.unsqueeze(-1) + p - mt)           # [B,H,C]\n\n        length = stop - start\n        lower = torch.ones(length, length, device=q.device, dtype=torch.bool).tril()\n        log_weights = (p.unsqueeze(-1) - p.unsqueeze(-2)\n                       + li.unsqueeze(-2) - mt.unsqueeze(-1))\n        weights = torch.exp(log_weights.clamp(max=0.0)).masked_fill(~lower, 0.0)\n        similarity = torch.matmul(qc, kc.transpose(-1, -2))\n        attention = similarity * weights\n        numerator = (torch.matmul(qc, c) * old_scale.unsqueeze(-1)\n                     + torch.matmul(attention, vc))\n        normalizer = (torch.sum(qc * n.unsqueeze(-2), dim=-1) * old_scale\n                      + attention.sum(dim=-1))\n        # Stabilized denominator from mLSTM. Clamping only guards exp overflow\n        # when a very long all-padding span is provided.\n        floor = torch.exp((-mt).clamp(max=30))\n        denominator = torch.maximum(normalizer.abs(), floor).clamp_min(1e-6)\n        outputs.append(numerator / denominator.unsqueeze(-1))\n\n        end_scale = old_scale[..., -1]\n        end_weights = torch.exp(\n            (p[..., -1:] - p + li - mt[..., -1:]).clamp(max=0.0)\n        )\n        c = c * end_scale[..., None, None] + torch.matmul(\n            kc.transpose(-1, -2), vc * end_weights.unsqueeze(-1)\n        )\n        n = n * end_scale[..., None] + (kc * end_weights.unsqueeze(-1)).sum(dim=-2)\n        m = mt[..., -1]\n    return torch.cat(outputs, dim=-2), CeNNXLSTMState(c, n, m, state.conv_cache)\n\n\ndef tokenwise_cenn_mlstm_reference(q: Tensor, k: Tensor, v: Tensor,\n                                   log_forget: Tensor, log_input: Tensor,\n                                   state: CeNNXLSTMState | None = None):\n    """Independent token-by-token numerical oracle, never use for fast inference."""\n    b, h, t, f = q.shape\n    if state is None:\n        state = initialize_state(b, h, f, v.shape[-1], device=q.device, dtype=q.dtype)\n    c, n, m = state.matrix, state.normalizer, state.log_scale\n    output = []\n    for i in range(t):\n        mm = torch.maximum(log_forget[:, :, i] + m, log_input[:, :, i])\n        forget = torch.exp(log_forget[:, :, i] + m - mm)\n        write = torch.exp(log_input[:, :, i] - mm)\n        c = (c * forget[..., None, None]\n             + k[:, :, i, :, None] * (v[:, :, i] * write[..., None])[:, :, None])\n        n = n * forget[..., None] + k[:, :, i] * write[..., None]\n        num = torch.einsum("bhf,bhfd->bhd", q[:, :, i], c)\n        denom_raw = torch.einsum("bhf,bhf->bh", q[:, :, i], n).abs()\n        denom = torch.maximum(denom_raw, torch.exp((-mm).clamp(max=30))).clamp_min(1e-6)\n        output.append(num / denom[..., None])\n        m = mm\n    return torch.stack(output, dim=-2), CeNNXLSTMState(c, n, m, state.conv_cache)\n\n\nclass GPUCeNNmLSTM(nn.Module):\n    """Depthwise cellular mixing + GPU-friendly matrix memory.\n\n    Causal mode supports bounded-state streaming with `return_state=True`.\n    Bidirectional mode runs two chunk scans and preserves encoder future context;\n    it does not support recurrent state API. Existing pretrained layers are NOT\n    numerically preserved on installation: distill and gate before replacement.\n    """\n\n    def __init__(self, config: CeNNXLSTMConfig):\n        super().__init__()\n        self.config = config\n        d, h, f, v = config.dim, config.heads, config.feature_dim, config.value_dim\n        self.norm = nn.LayerNorm(d)\n        self.cellular = nn.Conv1d(d, d, kernel_size=config.conv_kernel, groups=d, bias=True)\n        self.cellular_gain = nn.Parameter(torch.tensor(0.1))\n        self.qkv = nn.Linear(d, h * (f * 2 + v))\n        self.gates = nn.Linear(d, h * 2)\n        self.output = nn.Linear(h * v, d)\n        self.residual_gain = nn.Parameter(torch.tensor(-2.0))\n        nn.init.constant_(self.gates.bias[:h], 3.0)  # forget ≈ 0.95\n        nn.init.constant_(self.gates.bias[h:], -2.0) # restrained input updates\n\n    def _cellular(self, x: Tensor, state: CeNNXLSTMState | None):\n        d, kk = self.config.dim, self.config.conv_kernel - 1\n        xc = x.transpose(1, 2)\n        if self.config.bidirectional:\n            # Centered convolution for bidirectional ModernBERT-like encoders.\n            cells = self.cellular(F.pad(xc, (kk // 2, kk // 2)))\n            cache = None\n        else:\n            if state is None or state.conv_cache is None:\n                previous = xc.new_zeros(xc.shape[0], d, kk)\n            else:\n                previous = state.conv_cache\n                if previous.shape != (xc.shape[0], d, kk):\n                    raise ValueError("convolution cache shape mismatch")\n            cells = self.cellular(torch.cat([previous, xc], dim=-1))\n            cache = torch.cat([previous, xc], dim=-1)[:, :, -kk:].clone() if kk else xc.new_empty(xc.shape[0], d, 0)\n        return x + self.cellular_gain.tanh() * F.silu(cells.transpose(1, 2)), cache\n\n    def forward(self, x: Tensor, state: CeNNXLSTMState | None = None,\n                *, valid_mask: Tensor | None = None, return_state: bool = False):\n        if x.ndim != 3 or x.shape[-1] != self.config.dim or x.shape[1] < 1:\n            raise ValueError("expected x [B,T,dim] with T>0")\n        if self.config.bidirectional and (state is not None or return_state):\n            raise ValueError("bidirectional encoder mode has no streaming state API")\n        if valid_mask is not None and (valid_mask.shape != x.shape[:2] or valid_mask.dtype != torch.bool):\n            raise ValueError("valid_mask must be bool [B,T]")\n        source = self.norm(x)\n        if valid_mask is not None:\n            source = source.masked_fill(~valid_mask.unsqueeze(-1), 0)\n        features, cache = self._cellular(source, state)\n        b, t, _ = features.shape\n        h, f, dv = self.config.heads, self.config.feature_dim, self.config.value_dim\n        split = self.qkv(features).split((h*f, h*f, h*dv), dim=-1)\n        q = F.normalize(split[0].reshape(b, t, h, f), dim=-1).transpose(1, 2)\n        k = F.normalize(split[1].reshape(b, t, h, f), dim=-1).transpose(1, 2)\n        v = split[2].reshape(b, t, h, dv).transpose(1, 2)\n        gate_f, gate_i = self.gates(features).reshape(b, t, 2, h).unbind(dim=2)\n        log_f = F.logsigmoid(gate_f).transpose(1, 2)\n        log_i = gate_i.transpose(1, 2).clamp(-12, 8)\n        if valid_mask is not None:\n            pad = ~valid_mask[:, None]\n            log_f = log_f.masked_fill(pad, 0.0)\n            log_i = log_i.masked_fill(pad, float(\'-inf\'))\n        # Accumulate recurrent memory in FP32 for mixed-precision stability.\n        q, k, v, log_f, log_i = (z.float() for z in (q, k, v, log_f, log_i))\n        if state is not None:\n            recurrent = CeNNXLSTMState(state.matrix, state.normalizer, state.log_scale)\n        else:\n            recurrent = None\n        y, new_state = chunkwise_cenn_mlstm(q, k, v, log_f, log_i, recurrent,\n                                             chunk_size=self.config.chunk_size)\n        if self.config.bidirectional:\n            yr, _ = chunkwise_cenn_mlstm(q.flip(-2), k.flip(-2), v.flip(-2),\n                                         log_f.flip(-1), log_i.flip(-1),\n                                         chunk_size=self.config.chunk_size)\n            y = (y + yr.flip(-2)) * 0.5\n        merged = y.transpose(1, 2).reshape(b, t, h * dv)\n        output = x + torch.sigmoid(self.residual_gain) * self.output(merged.to(self.output.weight.dtype))\n        if valid_mask is not None:\n            output = output.masked_fill(~valid_mask.unsqueeze(-1), 0)\n        new_state.conv_cache = cache\n        return (output, new_state) if return_state else output\n\n    def recurrent_state_bytes(self, batch_size: int = 1) -> int:\n        c = self.config\n        recurrent = batch_size * c.heads * (c.feature_dim * c.value_dim + c.feature_dim + 1) * 4\n        conv = batch_size * c.dim * (c.conv_kernel - 1) * self.norm.weight.element_size()\n        return recurrent + conv\n',encoding='utf-8')
(MODULE_DIR/'cenn_x2_layer.py').write_text('"""CeNN-X2: multiscale cellular mixing + two-timescale chunkwise matrix memory.\n\nIndependent xLSTM-inspired experimental layer, not a pretrained xLSTM replica.\nOne batched recurrent memory call handles all timescales; causal mode supports\nstreaming and a bounded optional exact local SDPA branch. All accumulators use\nFP32. Training is parallel within chunks, with a Python loop *over chunks*.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\n\nfrom .cenn_xlstm_gpu import CeNNXLSTMState, chunkwise_cenn_mlstm\n\n\n@dataclass(frozen=True)\nclass CeNNX2Config:\n    dim: int = 256\n    heads: int = 4\n    key_dim: int = 32\n    value_dim: int = 64\n    chunk_size: int = 64\n    memory_scales: int = 2\n    conv_kernel: int = 5\n    conv_dilations: tuple[int, ...] = (1, 4)\n    local_window: int = 0\n    bidirectional: bool = False\n    residual_init: float = -3.0\n\n    def __post_init__(self):\n        if min(self.dim, self.heads, self.key_dim, self.value_dim, self.memory_scales) < 1:\n            raise ValueError(\'positive sizes required\')\n        if self.dim % self.heads:\n            raise ValueError(\'dim must be divisible by heads\')\n        if not 1 <= self.chunk_size <= 256:\n            raise ValueError(\'chunk_size must lie in [1,256]\')\n        if self.conv_kernel < 1 or self.conv_kernel % 2 == 0:\n            raise ValueError(\'conv_kernel must be positive odd\')\n        if not self.conv_dilations or any(d <= 0 for d in self.conv_dilations):\n            raise ValueError(\'conv_dilations must be positive, nonempty\')\n        if self.local_window < 0:\n            raise ValueError(\'local_window cannot be negative\')\n\n\n@dataclass\nclass CeNNX2State:\n    matrix: Tensor                    # [B, S*H, F, Dv], FP32\n    normalizer: Tensor                # [B, S*H, F], FP32\n    log_scale: Tensor                 # [B, S*H], FP32\n    conv_cache: Tensor                # [B, D, max_receptive_field - 1], input dtype\n    local_keys: Tensor | None = None  # [B,H,L,F], input dtype\n    local_values: Tensor | None = None # [B,H,L,Dv]\n    local_valid: Tensor | None = None # [B,L], bool\n\n    @property\n    def nbytes(self):\n        return sum(t.numel() * t.element_size() for t in (\n            self.matrix, self.normalizer, self.log_scale, self.conv_cache,\n            self.local_keys, self.local_values, self.local_valid\n        ) if t is not None)\n\n\ndef _safe_local_attention(q: Tensor, k: Tensor, v: Tensor, *, window: int,\n                          bidirectional: bool, chunk_size: int,\n                          valid: Tensor | None = None,\n                          past_keys: Tensor | None = None,\n                          past_values: Tensor | None = None,\n                          past_valid: Tensor | None = None\n                          ) -> tuple[Tensor, Tensor | None, Tensor | None, Tensor | None]:\n    """Bounded SDPA: query chunks, context <= C+2(W-1), no T-by-T matrix.\n\n    `window` is the number of causal keys, including the query. Bidirectional\n    mode attends up to window-1 neighbors on either side (including itself).\n    """\n    b, h, t, d = q.shape\n    if past_keys is not None:\n        if bidirectional:\n            raise ValueError(\'bidirectional mode cannot use a previous local cache\')\n        if past_values is None or past_valid is None:\n            raise ValueError(\'incomplete local cache\')\n        if (past_keys.shape[:2] != (b, h) or past_keys.shape[-1] != d\n            or past_values.shape[:3] != past_keys.shape[:3]\n            or past_valid.shape != (b, past_keys.shape[2])):\n            raise ValueError(\'bad local cache shape\')\n        cache_len = past_keys.shape[2]\n        keys = torch.cat((past_keys, k), dim=2)\n        values = torch.cat((past_values, v), dim=2)\n    else:\n        cache_len = 0\n        keys, values = k, v\n    all_valid = torch.ones(b, t, device=q.device, dtype=torch.bool) if valid is None else valid\n    if cache_len:\n        all_valid = torch.cat((past_valid, all_valid), dim=1)\n\n    result = []\n    for start in range(0, t, chunk_size):\n        stop = min(start + chunk_size, t)\n        begin_k = max(0, cache_len + start - window + 1)\n        end_k = min(keys.shape[2], cache_len + stop + (window - 1 if bidirectional else 0))\n        candidate = keys[:, :, begin_k:end_k]\n        cv = values[:, :, begin_k:end_k]\n        qpos = cache_len + torch.arange(start, stop, device=q.device)\n        kpos = torch.arange(begin_k, end_k, device=q.device)\n        offset = qpos[:, None] - kpos[None, :]\n        allowed = ((offset.abs() < window) if bidirectional\n                   else ((offset >= 0) & (offset < window)))\n        allowed = allowed[None, None] & all_valid[:, None, None, begin_k:end_k]\n        # Invalid *queries* can use their own key if valid is False; output is\n        # masked to zero by the caller, ensuring no all-false SDPA row NaNs.\n        # If every key is invalid (entire padding chunk), allow one dummy key.\n        fallback = ~allowed.any(dim=-1, keepdim=True)\n        # Never branch on a CUDA tensor: avoid a GPU->CPU synchronization.\n        one = (kpos[None, :] == begin_k)[None, None, :, :]\n        allowed = allowed | (fallback & one)\n        y = F.scaled_dot_product_attention(\n            q[:, :, start:stop], candidate, cv, attn_mask=allowed,\n            dropout_p=0.0, is_causal=False,\n        )\n        result.append(y)\n    if bidirectional:\n        return torch.cat(result, dim=2), None, None, None\n    keep = window - 1\n    if keep:\n        # clone prevents tiny cache views retaining an entire prefill allocation\n        return (torch.cat(result, dim=2), keys[:, :, -keep:].clone(),\n                values[:, :, -keep:].clone(), all_valid[:, -keep:].clone())\n    return (torch.cat(result, dim=2), keys[:, :, :0].clone(),\n            values[:, :, :0].clone(), all_valid[:, :0].clone())\n\n\nclass CeNNX2Layer(nn.Module):\n    """Hybrid CeNN xLSTM layer with exact chunk/streaming equivalence.\n\n    Memory matrix is scaled and normalized in FP32. Fast and slow gates use\n    *independent learned decay*, with shared Q/K/V and a learned per-token\n    router. Distinct dilation convolutions provide short/medium neighborhood\n    mixing, and optional local SDPA augments precise retrieval.\n\n    Output is residual-compatible [B,T,D]; it is NOT equivalent to a native\n    pretrained Transformer attention module without distillation and gating.\n    """\n    def __init__(self, config: CeNNX2Config):\n        super().__init__()\n        self.config = config\n        d, h, kd, vd, s = (config.dim, config.heads, config.key_dim,\n                           config.value_dim, config.memory_scales)\n        self.norm = nn.LayerNorm(d)\n        self.cellular = nn.ModuleList([\n            nn.Conv1d(d, d, config.conv_kernel, dilation=r, groups=d, bias=True)\n            for r in config.conv_dilations\n        ])\n        # Starts near identity but all branches learn, with a per-channel gain.\n        self.cellular_gain = nn.Parameter(torch.full((len(config.conv_dilations), d), -2.0))\n        self.qkv = nn.Linear(d, h * (2 * kd + vd))\n        self.forget = nn.Linear(d, h * s)\n        self.input_gate = nn.Linear(d, h)\n        self.time_router = nn.Linear(d, h * s)\n        self.local_router = nn.Linear(d, h) if config.local_window else None\n        self.output = nn.Linear(h * vd, d)\n        self.residual_gate = nn.Parameter(torch.full((d,), config.residual_init))\n        # Half-life separation at initialization: S=2 -> approx [0.86, 0.995].\n        values = torch.linspace(0.86, 0.995, s)\n        init = torch.logit(values).repeat_interleave(h)\n        with torch.no_grad():\n            self.forget.weight.zero_()\n            # memory-scale-major layout matches reshape [B,T,S,H].\n            self.forget.bias.copy_(init)\n            self.input_gate.bias.fill_(-2.0)\n            self.time_router.bias.zero_()\n            if self.local_router is not None:\n                self.local_router.bias.fill_(-2.0)\n\n    def _cache_length(self):\n        return (self.config.conv_kernel - 1) * max(self.config.conv_dilations)\n\n    def initial_state(self, batch: int, *, device, dtype) -> CeNNX2State:\n        c = self.config\n        sh = c.memory_scales * c.heads\n        return CeNNX2State(\n            torch.zeros(batch, sh, c.key_dim, c.value_dim, device=device, dtype=torch.float32),\n            torch.zeros(batch, sh, c.key_dim, device=device, dtype=torch.float32),\n            torch.zeros(batch, sh, device=device, dtype=torch.float32),\n            torch.zeros(batch, c.dim, self._cache_length(), device=device, dtype=dtype),\n        )\n\n    def _cellular(self, x: Tensor, state: CeNNX2State | None) -> tuple[Tensor, Tensor]:\n        xt = x.transpose(1, 2)\n        cache_length = self._cache_length()\n        if self.config.bidirectional:\n            joined = None\n            next_cache = xt.new_empty(x.shape[0], self.config.dim, 0)\n        else:\n            old = (torch.zeros(x.shape[0], self.config.dim, cache_length,\n                               device=x.device, dtype=x.dtype)\n                   if state is None else state.conv_cache)\n            if old.shape != (x.shape[0], self.config.dim, cache_length):\n                raise ValueError(\'bad convolution cache\')\n            joined = torch.cat((old, xt), dim=-1)\n            next_cache = joined[..., -cache_length:].clone() if cache_length else joined[..., :0].clone()\n        z = x\n        for i, (conv, dilation) in enumerate(zip(self.cellular, self.config.conv_dilations)):\n            if self.config.bidirectional:\n                padding = (self.config.conv_kernel - 1) * dilation // 2\n                spatial = conv(F.pad(xt, (padding, padding)))\n            else:\n                # All convolutions share one max-span cached prefix.\n                left = (self.config.conv_kernel - 1) * dilation\n                spatial = conv(joined[:, :, cache_length-left:])\n            z = z + torch.sigmoid(self.cellular_gain[i])[None, None] * F.silu(\n                spatial.transpose(1, 2)\n            )\n        return z, next_cache\n\n    def forward(self, x: Tensor, state: CeNNX2State | None = None,\n                *, valid_mask: Tensor | None = None, return_state: bool = False):\n        cfg = self.config\n        if x.ndim != 3 or x.shape[-1] != cfg.dim or x.shape[1] < 1:\n            raise ValueError(\'input must be [B,T,dim], T>=1\')\n        if cfg.bidirectional and (return_state or state is not None):\n            raise ValueError(\'bidirectional encoder has no streaming state\')\n        if valid_mask is not None and (valid_mask.shape != x.shape[:2] or valid_mask.dtype != torch.bool):\n            raise ValueError(\'valid_mask must have shape [B,T] and bool dtype\')\n        batch, t, _ = x.shape\n        if state is None:\n            state = self.initial_state(batch, device=x.device, dtype=x.dtype)\n        if (state.matrix.shape != (batch, cfg.heads * cfg.memory_scales, cfg.key_dim, cfg.value_dim)\n            or state.conv_cache.dtype != x.dtype or state.matrix.dtype != torch.float32\n            or state.matrix.device != x.device):\n            raise ValueError(\'invalid recurrent state\')\n        xn = self.norm(x)\n        if valid_mask is not None:\n            xn = xn.masked_fill(~valid_mask[..., None], 0)\n        feat, conv_cache = self._cellular(xn, state)\n        h, kd, vd, scales = cfg.heads, cfg.key_dim, cfg.value_dim, cfg.memory_scales\n        qq, kk, vv = self.qkv(feat).split((h*kd, h*kd, h*vd), dim=-1)\n        q = F.normalize(qq.reshape(batch, t, h, kd), dim=-1).transpose(1, 2)\n        k = F.normalize(kk.reshape(batch, t, h, kd), dim=-1).transpose(1, 2)\n        v = vv.reshape(batch, t, h, vd).transpose(1, 2)\n        forget = F.logsigmoid(self.forget(feat).reshape(batch, t, scales, h))\n        incoming = self.input_gate(feat).reshape(batch, t, h)\n        # Broadcast shared Q/K/V to scale-major, independently gated memory heads.\n        qrep = q[:, None].expand(-1, scales, -1, -1, -1).reshape(batch, scales*h, t, kd)\n        krep = k[:, None].expand(-1, scales, -1, -1, -1).reshape(batch, scales*h, t, kd)\n        vrep = v[:, None].expand(-1, scales, -1, -1, -1).reshape(batch, scales*h, t, vd)\n        log_f = forget.permute(0, 2, 3, 1).reshape(batch, scales*h, t)\n        log_i = incoming.transpose(1, 2)[:, None].expand(-1, scales, -1, -1).reshape(batch, scales*h, t)\n        if valid_mask is not None:\n            padding = ~valid_mask[:, None]\n            log_f = log_f.masked_fill(padding, 0.0)\n            log_i = log_i.masked_fill(padding, float(\'-inf\'))\n        in_state = CeNNXLSTMState(state.matrix, state.normalizer, state.log_scale)\n        y, new_mem = chunkwise_cenn_mlstm(qrep.float(), krep.float(), vrep.float(),\n                                           log_f.float(), log_i.float(), in_state,\n                                           chunk_size=cfg.chunk_size)\n        if cfg.bidirectional:\n            yr, _ = chunkwise_cenn_mlstm(\n                qrep.float().flip(2), krep.float().flip(2), vrep.float().flip(2),\n                log_f.float().flip(2), log_i.float().flip(2),\n                chunk_size=cfg.chunk_size)\n            y = 0.5 * (y + yr.flip(2))\n        # [B,S,H,T,V] -> [B,H,T,V] using query-conditioned time-scale routing.\n        ys = y.reshape(batch, scales, h, t, vd)\n        weights = self.time_router(feat).reshape(batch, t, scales, h)\n        weights = weights.softmax(dim=2).permute(0, 2, 3, 1)\n        merged = (ys * weights[..., None]).sum(dim=1)\n        if cfg.local_window:\n            local, new_keys, new_values, new_valid = _safe_local_attention(\n                q, k, v, window=cfg.local_window, bidirectional=cfg.bidirectional,\n                chunk_size=cfg.chunk_size, valid=valid_mask,\n                past_keys=(state.local_keys if not cfg.bidirectional else None),\n                past_values=(state.local_values if not cfg.bidirectional else None),\n                past_valid=(state.local_valid if not cfg.bidirectional else None),\n            )\n            gate = torch.sigmoid(self.local_router(feat)).transpose(1, 2).unsqueeze(-1)\n            merged = gate * local + (1 - gate) * merged.to(local.dtype)\n        else:\n            new_keys = new_values = new_valid = None\n        raw = merged.transpose(1, 2).reshape(batch, t, h * vd)\n        output = x + torch.sigmoid(self.residual_gate)[None, None] * self.output(\n            raw.to(self.output.weight.dtype)\n        )\n        if valid_mask is not None:\n            output = output.masked_fill(~valid_mask[..., None], 0)\n        next_state = CeNNX2State(new_mem.matrix, new_mem.normalizer,\n                                new_mem.log_scale, conv_cache, new_keys, new_values, new_valid)\n        return (output, next_state) if return_state else output\n\n    def state_bytes(self, batch: int = 1, dtype: torch.dtype = torch.float32) -> int:\n        c = self.config\n        count = batch * c.heads * c.memory_scales * (c.key_dim*c.value_dim + c.key_dim + 1)\n        n = count * 4 + batch*c.dim*self._cache_length()*torch.empty((), dtype=dtype).element_size()\n        if c.local_window:\n            n += batch*(c.local_window-1)*(c.heads*(c.key_dim+c.value_dim)*torch.empty((),dtype=dtype).element_size()+1)\n        return n\n',encoding='utf-8')
sys.path.insert(0,str(MODULE_ROOT))
from cenn_x2_embedded.cenn_x2_layer import CeNNX2Layer, CeNNX2Config
print('Embedded CeNN-X2 source ready:', MODULE_DIR)

In [ ]:
# Small numerical smoke test, independent of loading Qwen weights.
torch.manual_seed(SEED)
toy=CeNNX2Layer(CeNNX2Config(dim=32,heads=4,key_dim=8,value_dim=8,
                            chunk_size=7,local_window=5,conv_dilations=(1,3))).to(DEVICE).eval()
toy_x=torch.randn(1,19,32,device=DEVICE)
with torch.inference_mode():
    a=toy(toy_x)
    st=None; outs=[]
    for start,stop in [(0,1),(1,4),(4,13),(13,19)]:
        z,st=toy(toy_x[:,start:stop],state=st,return_state=True)
        outs.append(z)
    difference=(a-torch.cat(outs,dim=1)).abs().max().item()
assert difference < 1e-4, difference
print('CeNN-X2 CUDA streaming contract PASSED; max error:', difference)
del toy,toy_x

## 2. Load original Qwen, inspect all layers
The repository checkpoint is multimodal, but `Qwen3_5ForCausalLM` loads the **text backbone** for this controlled text-only experiment. The code verifies actual loaded layer types instead of assuming model internals.

In [ ]:
from huggingface_hub import model_info
info=model_info(MODEL_ID)
REVISION=info.sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
expected_full=[3,7,11,15,19,23]
assert cfg.model_type=='qwen3_5_text',cfg.model_type
assert [i for i,t in enumerate(cfg.layer_types) if t=='full_attention']==expected_full
assert cfg.num_hidden_layers==24 and cfg.hidden_size==1024
assert all(0 <= i < cfg.num_hidden_layers for i in CANDIDATE_LAYERS)
print('Pinned revision:',REVISION,'model hidden dim:',cfg.hidden_size)
print('Candidates:',[(i,cfg.layer_types[i]) for i in CANDIDATE_LAYERS])
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(
    MODEL_ID,revision=REVISION,dtype=DTYPE,attn_implementation='sdpa',
    low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False)
model.config.use_cache=False
LAYERS=model.model.layers
for i in CANDIDATE_LAYERS:
    block=LAYERS[i]
    assert block.block_type == cfg.layer_types[i]
    name='self_attn' if block.block_type=='full_attention' else 'linear_attn'
    assert hasattr(block,name),(i,block.block_type)
    print(f'{i:2d} {block.block_type:18} {name:12} {sum(p.numel() for p in getattr(block,name).parameters()):,} params')
print('Loaded model parameters:',sum(p.numel() for p in model.parameters()))

## 3. Load **disjoint** calibration / validation / audit text
WikiText-2 `train`, `validation`, and `test` are independent. We generate fixed-length **unpadded** contiguous token blocks, since the CeNN-X2 research graft currently has no support for Qwen attention masks or multimodal caches. Training/evaluation are **full-prefill**, not token-by-token cached decoding.

In [ ]:
from datasets import load_dataset

def load_fixed_windows(split, limit, length):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    token_stream=[]
    for item in ds:
        text=item['text'].strip()
        if not text: continue
        token_stream.extend(tokenizer.encode(text,add_special_tokens=False))
        token_stream.append(tokenizer.eos_token_id)
        if len(token_stream) >= (limit+1)*length: break
    n=min(limit,len(token_stream)//length)
    if n < limit: raise RuntimeError(f'{split}: only {n} full blocks (asked {limit})')
    blocks=torch.tensor(token_stream[:limit*length],dtype=torch.long).view(limit,length)
    return blocks

TRAIN_IDS=load_fixed_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV_IDS=load_fixed_windows('validation',DEV_BLOCKS,SEQ_LEN)
TEST_IDS=load_fixed_windows('test',TEST_BLOCKS,SEQ_LEN)
fingerprints=[hashlib.sha256(x.numpy().tobytes()).hexdigest() for x in (TRAIN_IDS,DEV_IDS,TEST_IDS)]
assert len(set(fingerprints))==3
print('Split shapes:',TRAIN_IDS.shape,DEV_IDS.shape,TEST_IDS.shape)
print('SHA256 per partition:',fingerprints)
print('Calibration example:\n',tokenizer.decode(TRAIN_IDS[0][:48]))

## 4. Capture the **native** token-mixer IO
This is a local teacher-student experiment: record **exact pre-mixer hidden states and mixer output** for each candidate in **one frozen original Qwen**. Each candidate then trains without gradients through the 0.8B teacher. Captured activation tensors are stored in FP16 on CPU.

In [ ]:
MODULES={i:('self_attn' if LAYERS[i].block_type=='full_attention' else 'linear_attn') for i in CANDIDATE_LAYERS}
TEACHER={i:{'train':{'x':[],'y':[]},'dev':{'x':[],'y':[]}} for i in CANDIDATE_LAYERS}
_handles=[]

def make_hook(layer_idx,which):
    def hook(mod,args,kwargs,output):
        x=kwargs.get('hidden_states',args[0] if args else None)
        if x is None: raise RuntimeError(f'No input to layer {layer_idx}')
        y=output[0] if isinstance(output,tuple) else output
        TEACHER[layer_idx][which]['x'].append(x.detach().to('cpu',dtype=torch.float16))
        TEACHER[layer_idx][which]['y'].append(y.detach().to('cpu',dtype=torch.float16))
    return hook

@torch.inference_mode()
def prefill(ids,*,keep=1):
    return model(input_ids=ids.to(DEVICE),use_cache=False,logits_to_keep=keep)

for which,blocks in [('train',TRAIN_IDS),('dev',DEV_IDS)]:
    _handles=[]
    for idx,attribute in MODULES.items():
        _handles.append(getattr(LAYERS[idx],attribute).register_forward_hook(make_hook(idx,which),with_kwargs=True))
    try:
        for start in range(0,len(blocks),BATCH_SIZE):
            prefill(blocks[start:start+BATCH_SIZE])
    finally:
        for h in _handles: h.remove()
    for i in CANDIDATE_LAYERS:
        TEACHER[i][which]={k:torch.cat(v,dim=0).contiguous() for k,v in TEACHER[i][which].items()}
        print(f'{which:5} layer={i:2} captured x/y:',TEACHER[i][which]['x'].shape,
              'output RMS',round(float(TEACHER[i][which]['y'].float().square().mean().sqrt()),5))
if DEVICE.type=='cuda': torch.cuda.empty_cache()

## 5. CeNN-X2 **mixer-only** replacement and local calibration
`CeNNX2Layer` itself returns `x + update`, but **Qwen's decoder adds its own residual**. The adapter exposes **`CeNNX2Layer(x) - x`** so the decoder residual is **not added twice**. Linear-attention modules must return a tensor; full-attention modules must return `(tensor, attention_weights)`. The replacement explicitly rejects native KV/recurrent cache to prevent misleading generation results.

Training uses native **teacher mixer output**, normalized MSE plus cosine loss and a held-out local validation set. Native model weights stay frozen.

In [ ]:
import torch.nn as nn

class QwenCeNNX2Mixer(nn.Module):
    def __init__(self, layer: CeNNX2Layer, block_type: str):
        super().__init__()
        assert block_type in ('full_attention','linear_attention')
        self.cenn=layer
        self.block_type=block_type

    def delta(self,x):
        # Train in FP32; during Qwen evaluation, AMP casts FP32 projection weights
        # to the model's device dtype while matrix memories stay FP32.
        y=self.cenn(x.float())
        return (y-x.float()).to(dtype=x.dtype)

    def forward(self,hidden_states,attention_mask=None,past_key_values=None,
                cache_params=None,**kwargs):
        if past_key_values is not None or cache_params is not None:
            raise RuntimeError('CeNN-X2 graft supports use_cache=False only')
        # All provided evaluation examples are equal-length *un-padded* blocks.
        # A 4D causal additive mask on full-attention layers is handled by the
        # causal CeNN-X2 architecture; 2D padding is forbidden here.
        if attention_mask is not None and attention_mask.ndim == 2:
            if not bool(torch.all(attention_mask != 0)):
                raise ValueError('No padded inputs supported by experimental Qwen graft')
        delta=self.delta(hidden_states)
        return (delta,None) if self.block_type=='full_attention' else delta


def make_candidate(idx,variant):
    assert variant in VARIANTS
    params=dict(dim=cfg.hidden_size,heads=4,key_dim=32,value_dim=64,
                chunk_size=CHUNK_SIZE,conv_dilations=(1,4),
                conv_kernel=5,local_window=0,residual_init=0.0)
    if variant=='single':
        params.update(memory_scales=1,conv_dilations=(1,))
    elif variant=='dual':
        params.update(memory_scales=2)
    elif variant=='dual_local':
        params.update(memory_scales=2,local_window=32)
    return QwenCeNNX2Mixer(CeNNX2Layer(CeNNX2Config(**params)).to(DEVICE),
                           LAYERS[idx].block_type)


def loss_and_stats(candidate,x,y):
    pred=candidate.delta(x).float()
    target=y.float()
    mse=(pred-target).square().mean()
    energy=target.square().mean().clamp_min(1e-8)
    nmse=mse/energy
    cosine=F.cosine_similarity(pred.reshape(-1,pred.shape[-1]),
                               target.reshape(-1,target.shape[-1]),dim=-1).mean()
    return nmse + 0.15*(1-cosine),nmse,cosine

@torch.no_grad()
def local_eval(candidate,idx,which='dev'):
    candidate.eval()
    xs,ys=TEACHER[idx][which]['x'],TEACHER[idx][which]['y']
    total=0; sq_error=0; sq_target=0; cosine_total=0
    for start in range(0,len(xs),BATCH_SIZE):
        xx=xs[start:start+BATCH_SIZE].to(DEVICE,dtype=DTYPE)
        yy=ys[start:start+BATCH_SIZE].to(DEVICE,dtype=DTYPE)
        with torch.autocast('cuda',dtype=DTYPE,enabled=DEVICE.type=='cuda'):
            pred=candidate.delta(xx).float()
        target=yy.float()
        n=target.shape[0]*target.shape[1]
        sq_error+=(pred-target).square().sum().item()
        sq_target+=target.square().sum().item()
        cosine_total+=F.cosine_similarity(pred,target,dim=-1).sum().item()
        total+=n
    return dict(nmse=sq_error/max(sq_target,1e-8),cosine=cosine_total/max(total,1))


def fit_candidate(idx,variant,steps=TRAIN_STEPS):
    adapter=make_candidate(idx,variant)
    adapter.train()
    optimizer=torch.optim.AdamW(adapter.parameters(),lr=LR,weight_decay=0.01)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16 and DEVICE.type=='cuda'))
    xs=TEACHER[idx]['train']['x'];ys=TEACHER[idx]['train']['y']
    best_score=float('inf');best_state=None;best_dev=None
    for step in range(1,steps+1):
        indices=torch.randint(0,len(xs),(BATCH_SIZE,))
        x=xs[indices].to(DEVICE,dtype=DTYPE)
        y=ys[indices].to(DEVICE,dtype=DTYPE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE,enabled=DEVICE.type=='cuda'):
            loss,_,_=loss_and_stats(adapter,x,y)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'Nonfinite training loss on layer {idx} / {variant}')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(adapter.parameters(),1.)
        scaler.step(optimizer);scaler.update()
        if step==1 or step==steps or step%max(1,steps//4)==0:
            stats=local_eval(adapter,idx)
            score=stats['nmse'] + 0.15*(1-stats['cosine'])
            if score < best_score:
                best_score=score;best_state={k:v.detach().cpu().clone() for k,v in adapter.state_dict().items()}
                best_dev=stats
            print(f'layer={idx:2d} {variant:10s} step={step:3d} '
                  f'loss={loss.item():.4f} val_nmse={stats["nmse"]:.4f} '
                  f'val_cos={stats["cosine"]:.4f}',flush=True)
    adapter.load_state_dict(best_state);adapter.eval()
    return adapter,best_dev

from contextlib import contextmanager
@contextmanager
def substituted(idx, candidate):
    attribute=MODULES[idx]
    original=getattr(LAYERS[idx],attribute)
    assert original is not candidate
    setattr(LAYERS[idx],attribute,candidate)
    try:
        yield
    finally:
        setattr(LAYERS[idx],attribute,original)

## 6. Model-level evaluation (no training, no cache)
For each student candidate we evaluate **same unpadded validation token blocks**. Scores use selected next-token logits at specified token positions, with native Qwen outputs cached once. The same teacher logits, tokens, and model instance are reused for all variants. The **test split is not read until the final audit**.

In [ ]:
EVAL_POSITIONS=sorted(set(p for p in [15,31,47,SEQ_LEN-2] if 0 <= p < SEQ_LEN-1))
print('Evaluating next-token positions:',EVAL_POSITIONS)

@torch.inference_mode()
def score_logits(blocks, teacher=None):
    total=0; nll_sum=0; top_sum=0; kl_sum=0; agreement=0
    refs=[]
    for start in range(0,len(blocks),BATCH_SIZE):
        chunk=blocks[start:start+BATCH_SIZE]
        logits=prefill(chunk,keep=0).logits[:,EVAL_POSITIONS].float()
        tgt=chunk[:,[p+1 for p in EVAL_POSITIONS]].to(DEVICE)
        lp=F.log_softmax(logits,dim=-1)
        nll=(-lp.gather(-1,tgt.unsqueeze(-1)).squeeze(-1)).sum().item()
        nll_sum+=nll
        total+=tgt.numel()
        prediction=logits.argmax(-1)
        top_sum+=(prediction==tgt).sum().item()
        if teacher is None:
            refs.append(logits.to('cpu',dtype=torch.float16))
        else:
            tlogits=teacher[start//BATCH_SIZE].to(DEVICE,dtype=torch.float32)
            tlp=F.log_softmax(tlogits,dim=-1)
            kl_sum+=(tlp.exp()*(tlp-lp)).sum(dim=-1).sum().item()
            agreement+=(prediction==tlogits.argmax(-1)).sum().item()
    result=dict(nll=nll_sum/total,next_token_accuracy=top_sum/total)
    if teacher is not None:
        result.update(teacher_kl=kl_sum/total,teacher_top1_agreement=agreement/total)
    return result,refs

BASE_DEV,TEACHER_DEV_LOGITS=score_logits(DEV_IDS)
print('Native validation:',BASE_DEV)

@torch.inference_mode()
def benchmark_prefill(ids,repeats=5,warmups=2):
    # identical model, inputs, dtype, and use_cache=False for native/student
    sample=ids[:min(BATCH_SIZE,len(ids))].to(DEVICE)
    timings=[]
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
    for r in range(warmups+repeats):
        if DEVICE.type=='cuda':
            torch.cuda.synchronize()
            start=torch.cuda.Event(enable_timing=True);stop=torch.cuda.Event(enable_timing=True)
            start.record()
            with torch.autocast('cuda',dtype=DTYPE,enabled=DEVICE.type=='cuda'):
                result=model(input_ids=sample,use_cache=False,logits_to_keep=1)
            stop.record();torch.cuda.synchronize();duration=start.elapsed_time(stop)
        else:
            start=time.perf_counter();result=model(input_ids=sample,use_cache=False,logits_to_keep=1)
            duration=(time.perf_counter()-start)*1000
        assert bool(torch.isfinite(result.logits).all()),'Nonfinite model logits'
        if r>=warmups:timings.append(duration)
    mem=(torch.cuda.max_memory_allocated()/(1024**2)) if DEVICE.type=='cuda' else None
    return dict(latency_ms=statistics.median(timings),peak_cuda_allocated_mib=mem)

BASE_TIMING=benchmark_prefill(DEV_IDS)
print('Native whole-model prefill:',BASE_TIMING)

## 7. Scan middle layers and three CeNN-X2 configurations
Default = **8 layers × 3 architectures = 24 independent teacher-to-student calibrations** (24 steps each under `quick`). This can be compute-intensive, especially with the native Qwen3.5 model resident. To cut runtime, edit `CANDIDATE_LAYERS` / `VARIANTS` before running from the top. The test split remains sealed for the later winner-only audit.

In [ ]:
RESULTS=[]
WEIGHTS_DIR=OUTPUT_DIR/'weights'
WEIGHTS_DIR.mkdir(exist_ok=True)
for layer_idx in CANDIDATE_LAYERS:
    for variant in VARIANTS:
        torch.manual_seed(SEED+layer_idx*13+VARIANTS.index(variant))
        print(f'\n========== layer={layer_idx}, type={cfg.layer_types[layer_idx]}, variant={variant} ==========')
        adapter,local=fit_candidate(layer_idx,variant)
        with substituted(layer_idx,adapter):
            dev,_=score_logits(DEV_IDS,teacher=TEACHER_DEV_LOGITS)
            timing=benchmark_prefill(DEV_IDS,repeats=3,warmups=1)
        model_params=sum(x.numel() for x in adapter.parameters())
        record=dict(layer=layer_idx,layer_type=cfg.layer_types[layer_idx],variant=variant,
                    train_steps=TRAIN_STEPS,adapter_parameters=model_params,
                    local_dev_nmse=local['nmse'],local_dev_cosine=local['cosine'],
                    dev_nll=dev['nll'],dev_nll_delta=dev['nll']-BASE_DEV['nll'],
                    dev_kl=dev['teacher_kl'],dev_agreement=dev['teacher_top1_agreement'],
                    dev_gold_acc=dev['next_token_accuracy'],
                    latency_ms=timing['latency_ms'],
                    prefill_speedup=BASE_TIMING['latency_ms']/timing['latency_ms'],
                    peak_cuda_allocated_mib=timing['peak_cuda_allocated_mib'])
        RESULTS.append(record)
        fn=WEIGHTS_DIR/f'cenn_x2_layer{layer_idx}_{variant}.pt'
        torch.save({'layer':layer_idx,'variant':variant,'cenn_config':asdict(adapter.cenn.config),
                    'adapter_state_dict':{k:v.detach().cpu() for k,v in adapter.state_dict().items()},
                    'model_id':MODEL_ID,'revision':REVISION,'use_cache':False},fn)
        pd.DataFrame(RESULTS).to_csv(OUTPUT_DIR/'results_validation.csv',index=False)
        print('CANDIDATE:',json.dumps(record,indent=2))
        del adapter
        gc.collect()
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
print('COMPLETED:',len(RESULTS),'candidate experiments')

## 8. Rank candidates on **validation only**
Do not pick the fastest broken model. Gate by task fidelity, then rank eligible variants by measured speed. The thresholds below are research screening criteria rather than published acceptance standards. When no candidate passes, **retain original Qwen** and optionally inspect the least damaging experimental alternative without calling it successful.

In [ ]:
frame=pd.DataFrame(RESULTS)
# Explicit fidelity acceptance criteria; tune on validation, do not tune on final test.
MAX_DELTA_NLL=0.15
MIN_TEACHER_AGREEMENT=0.90
MIN_LOCAL_COSINE=0.95
frame['quality_gate']=(
    (frame.dev_nll_delta<=MAX_DELTA_NLL) &
    (frame.dev_agreement>=MIN_TEACHER_AGREEMENT) &
    (frame.local_dev_cosine>=MIN_LOCAL_COSINE)
)
frame=frame.sort_values(['quality_gate','dev_nll_delta','dev_kl'],ascending=[False,True,True])
display(frame[['layer','layer_type','variant','local_dev_cosine','local_dev_nmse',
               'dev_nll_delta','dev_agreement','prefill_speedup','quality_gate']])
eligible=frame[frame.quality_gate]
if len(eligible):
    winner=eligible.sort_values(['prefill_speedup','dev_nll_delta'],ascending=[False,True]).iloc[0]
    accepted=True
else:
    winner=frame.sort_values(['dev_nll_delta','dev_kl']).iloc[0]
    accepted=False
print('Selected for **diagnostic test audit**:',winner[['layer','variant','quality_gate']].to_dict())
print('Quality accepted:',accepted,'(if False, keep native model)')
plt.figure(figsize=(11,5))
labels=[f'L{r.layer}:{r.variant}' for r in frame.itertuples()]
plt.bar(range(len(frame)),frame.dev_nll_delta)
plt.axhline(MAX_DELTA_NLL,linestyle='--',label='dev NLL gate')
plt.xticks(range(len(frame)),labels,rotation=80)
plt.ylabel('Validation NLL delta vs native (lower better)')
plt.legend();plt.tight_layout();plt.show()

## 9. One **held-out test audit** on the selected candidate
The test set has not been read by calibration, hyperparameter choice, or validation ranking. Record one native Qwen test baseline and one chosen student test. If the validation gate fails, the audit is still diagnostic but **not an accepted replacement**.

In [ ]:
WIN_LAYER=int(winner.layer)
WIN_VARIANT=str(winner.variant)
WIN_PATH=WEIGHTS_DIR/f'cenn_x2_layer{WIN_LAYER}_{WIN_VARIANT}.pt'
payload=torch.load(WIN_PATH,map_location='cpu',weights_only=True)
winning_adapter=make_candidate(WIN_LAYER,WIN_VARIANT)
winning_adapter.load_state_dict(payload['adapter_state_dict'])
winning_adapter.eval()
BASE_TEST,TEST_REFERENCE=score_logits(TEST_IDS)
BASE_TEST_LATENCY=benchmark_prefill(TEST_IDS,repeats=5,warmups=2)
with substituted(WIN_LAYER,winning_adapter):
    STUDENT_TEST,_=score_logits(TEST_IDS,teacher=TEST_REFERENCE)
    STUDENT_TEST_LATENCY=benchmark_prefill(TEST_IDS,repeats=5,warmups=2)
TEST_DELTA_NLL=STUDENT_TEST['nll']-BASE_TEST['nll']
TEST_SPEEDUP=BASE_TEST_LATENCY['latency_ms']/STUDENT_TEST_LATENCY['latency_ms']
FINAL_PASS=bool(accepted and TEST_DELTA_NLL<=MAX_DELTA_NLL
                and STUDENT_TEST['teacher_top1_agreement']>=MIN_TEACHER_AGREEMENT
                and TEST_SPEEDUP>1.0)
summary={
    'model':MODEL_ID,'model_revision':REVISION,'profile':PROFILE,
    'candidate_layers':CANDIDATE_LAYERS,'variants':VARIANTS,
    'data':'Salesforce/wikitext/wikitext-2-raw-v1','fingerprints':fingerprints,
    'quality_thresholds':{'max_delta_nll':MAX_DELTA_NLL,
         'min_teacher_agreement':MIN_TEACHER_AGREEMENT,'min_local_cosine':MIN_LOCAL_COSINE},
    'selection_accepted':accepted,'final_quality_and_speed_pass':FINAL_PASS,'selected':{'layer':WIN_LAYER,'variant':WIN_VARIANT},
    'native_validation':BASE_DEV,'native_validation_timing':BASE_TIMING,
    'native_test':BASE_TEST,'native_test_timing':BASE_TEST_LATENCY,
    'student_test':STUDENT_TEST,'student_test_timing':STUDENT_TEST_LATENCY,
    'test_delta_nll':TEST_DELTA_NLL,
    'test_prefill_speedup':TEST_SPEEDUP,
    'all_validation_results':RESULTS,
    'limitations':'No cached generation or vision testing; ref CeNN-X2 Python chunk loops; no GPU results pre-populated.'
}
(OUTPUT_DIR/'final_audit.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
print(json.dumps({k:summary[k] for k in ['selection_accepted','final_quality_and_speed_pass','selected','native_test','student_test',
             'test_delta_nll','test_prefill_speedup']},indent=2))
if not FINAL_PASS: print('NOT ACCEPTED as a quality-preserving speedup: keep native Qwen3.5.')

## 10. Optional qualitative continuation (cache-free full prefill)
This is a **greedy teacher-forced full-prefix re-evaluation** (recomputing the entire prompt each token); it is intentionally not `.generate()` and **cannot** be used for meaningful decode speed. Strictly for qualitative side-by-side outputs.

In [ ]:
PROMPTS=[
    'The capital of Austria is',
    'A robotic swarm can navigate a city safely when',
    'The main difference between linear attention and full attention is',
]

@torch.inference_mode()
def slow_no_cache_greedy(prompt,extra=24):
    tokens=tokenizer(prompt,return_tensors='pt',add_special_tokens=True).input_ids.to(DEVICE)
    for _ in range(extra):
        logits=model(input_ids=tokens,use_cache=False,logits_to_keep=1).logits[:,-1,:]
        nxt=logits.argmax(-1,keepdim=True)
        tokens=torch.cat([tokens,nxt],dim=1)
        if int(nxt.item()) == tokenizer.eos_token_id:break
    return tokenizer.decode(tokens[0],skip_special_tokens=True)

for prompt in PROMPTS:
    print('PROMPT:',prompt)
    base=slow_no_cache_greedy(prompt,extra=16)
    with substituted(WIN_LAYER,winning_adapter):
        student=slow_no_cache_greedy(prompt,extra=16)
    print('NATIVE:',base,'\nCeNN-X2:',student,'\n')

## 11. Package result files (notebooks / weights / tables)
All results are generated **when you actually run the notebook**. They are not reported as benchmark wins in advance. Use `/content/qwen35_cenn_x2_middle_ablation/` for Colab artifacts, then download a ZIP.

In [ ]:
import shutil
manifest={
    'model_id':MODEL_ID,'model_revision':REVISION,
    'notebook':'Qwen35_08B_CeNN_X2_Middle_Layer_Ablation_Colab.ipynb',
    'adapter_checkpoint_only':True,'original_checkpoint_unchanged':True,
    'cache_compatible':False,'multimodal_supported':False,
    'train_steps':TRAIN_STEPS,'quality_gate_on_validation':True,
}
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive(str(OUTPUT_DIR),'zip',root_dir=OUTPUT_DIR)
print('Results archive:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Download manually in a non-Colab environment.')

### What would count as a real improvement?

An acceptable replacement should pass **both** the pre-declared validation gate and a clean held-out test audit, with near-native next-token NLL and teacher decisions, *plus* a measurable reduction in **whole-model prefill** latency (and eventually **cached decode**, after cache API integration). Improving an isolated mixer microbenchmark without preserving model quality is insufficient.

For incremental research, investigate **L10 / L12 / L14 linear mixers** first, then **L11 / L15 full attention**. These are candidate positions, not empirical winners.

**References:** [Qwen3.5-0.8B configuration](https://huggingface.co/Qwen/Qwen3.5-0.8B/blob/main/config.json) · [Transformers Qwen3.5 implementation](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_5/modeling_qwen3_5.py) · [CeNN-X2 prototype](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/CeNN_X2_DualTimescale_GPU_Colab.ipynb)